In [1]:
question = 'How many users are there average trades per user?'
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

import chromadb
import json

from pprint import pprint

# from .prompts import SYSTEM_PROMPT

# Semantic Layer (RAG)

## Retriever

### 1. Embeddings

In [2]:
# app/rag/embeddings.py

from openai import OpenAI
import json

client = OpenAI()


def load_schema_docs(path="data/schema_docs.json"):
    with open(path, "r") as f:
        return json.load(f)


def format_doc(doc):
    # NEW: handle metrics
    if doc.get("type") == "metric":
        return f"""
        Metric: {doc['name']}
        Definition: {doc['definition']}
        """

    # existing behavior (tables)
    return f"""
    Table: {doc['table']}
    Description: {doc['description']}
    Columns: {', '.join(doc['columns'])}
    """


def generate_embeddings(docs):
    texts = [format_doc(doc) for doc in docs]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    embeddings = [e.embedding for e in response.data]

    return list(zip(texts, embeddings))


In [3]:
# app/rag/embeddings.py
# load_schema_docs
import json
def load_schema_docs(path="data/schema_docs.json"):
    with open(path, "r") as f:
        return json.load(f)
load_schema_docs()

[{'table': 'users',
  'description': 'Stores user account information',
  'columns': ['user_id', 'signup_date']},
 {'table': 'athletes',
  'description': 'Stores athlete profiles',
  'columns': ['athlete_id', 'name', 'sport', 'team']},
 {'table': 'trades',
  'description': 'Records all trades made by users for athletes',
  'columns': ['trade_id', 'user_id', 'athlete_id', 'amount', 'created_at']},
 {'type': 'metric',
  'name': 'average trades per user',
  'definition': 'total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades'}]

In [5]:

temp_doc = load_schema_docs()
[format_doc(doc) for doc in temp_doc]

['\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ',
 '\n    Table: athletes\n    Description: Stores athlete profiles\n    Columns: athlete_id, name, sport, team\n    ',
 '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
 '\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ']

In [ ]:
from app.rag.embeddings import load_schema_docs, generate_embeddings

# clinet is OpenAI

temp_doc = load_schema_docs()
embedded_docs = generate_embeddings(temp_doc)

embedded_docs

[('\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ',
  [-0.0038547515869140625,
   0.0140838623046875,
   0.0084075927734375,
   0.01776123046875,
   0.07147216796875,
   -0.01345062255859375,
   0.0262603759765625,
   -0.028411865234375,
   -0.0161285400390625,
   -0.0186004638671875,
   0.004947662353515625,
   -0.0232696533203125,
   0.037109375,
   -0.0289306640625,
   -0.00426483154296875,
   0.0212249755859375,
   -0.038482666015625,
   0.04315185546875,
   -0.024627685546875,
   0.044158935546875,
   -0.006916046142578125,
   -0.005218505859375,
   0.05267333984375,
   -0.017669677734375,
   0.05401611328125,
   0.08642578125,
   -0.030426025390625,
   0.03375244140625,
   0.0023860931396484375,
   -0.048095703125,
   0.00757598876953125,
   -0.0112762451171875,
   0.01406097412109375,
   0.040313720703125,
   0.0286407470703125,
   -0.0306243896484375,
   0.035186767578125,
   0.01016998291015625,
   0.00482940673828

### 2. Chroma

In [7]:
# app/rag/vector_store.py

import chromadb
# from chromadb.config import Settings

# get_collection() → client created
# store_embeddings() → upsert() → writes to disk → folder created

def get_chroma_client():
    """
    Create a persistent Chroma client.

    Why:
    - Ensures embeddings are saved to disk (./chroma_db)
    - Fixes issue where no folder was created
    """
    return chromadb.PersistentClient(path="./chroma_db")

def get_collection(name="schema_docs"):
    """
    Get or create a collection.

    Why:
    - Collections group related embeddings
    - 'schema_docs' will store your table metadata
    """ 
    client = get_chroma_client()
    return client.get_or_create_collection(name=name)

def store_embeddings(collection, embedded_docs):
    """
    Store embeddings in Chroma.

    Input:
    - embedded_docs = [(text, embedding_vector)]

    Why:
    - This makes your schema searchable via similarity
    """
    texts = [doc[0] for doc in embedded_docs]
    embeddings = [doc[1] for doc in embedded_docs]
    ids = [f"id_{i}" for i in range(len(texts))]

    collection.upsert(
        documents=texts,
        embeddings=embeddings,
        ids=ids
    )
    
def query_collection(collection, query_embedding, n_results=2):
    """
    Query similar documents.

    Why:
    - This is the core of RAG retrieval
    - Finds relevant tables based on meaning (not keywords)
    """
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

### 3. Retriever

In [8]:
# app/rag/retriever.py

client = OpenAI()
from app.rag.vector_store import get_collection, query_collection

def retrieve_relevant_docs(question: str, top_k: int = 4):
    """
    Improved retrieval:
    - Always try to match metrics first (deterministic)
    - Then use vector search for tables
    """

    # -------------------------
    # 1. Load all docs (for metrics)
    # -------------------------
    # from app.rag.embeddings import load_schema_docs, format_doc

    raw_docs = load_schema_docs()

    metric_docs = []
    for doc in raw_docs:
        if doc.get("type") == "metric":
            if doc["name"].lower() in question.lower():
                metric_docs.append(format_doc(doc))
                
    # -------------------------
    # 2. Vector search (tables)
    # -------------------------
    embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    collection = get_collection()
    results = query_collection(collection, embedding, n_results=top_k)

    docs = results.get("documents", [[]])[0]

    table_docs = [d for d in docs if "Table:" in d]

    # -------------------------
    # 3. Remove weak matches
    # -------------------------
    filtered_tables = []
    for t in table_docs:
        if any(word in t.lower() for word in question.lower().split()):
            filtered_tables.append(t)

    if not filtered_tables:
        filtered_tables = table_docs

    # -------------------------
    # 4. Final ordering
    # -------------------------
    return metric_docs + filtered_tables
retrieve_relevant_docs(question=question,top_k=4)


['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ',
 '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
 '\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ']

## Debug

In [32]:
# app/rag/retriever.py

client = OpenAI()

# retrieve_relevant_docs(question: str, top_k: int = 4):
"""
Improved retrieval:
- Always try to match metrics first (deterministic)
- Then use vector search for tables
"""
# -------------------------
# 1. Load all docs (for metrics)
# -------------------------

## raw_docs = load_schema_docs()
    # from app.rag.embeddings import load_schema_docs
path="data/schema_docs.json"
with open(path, "r") as f:
    raw_docs = json.load(f)

print('output of load_schema_docs:','\n',raw_docs)

    # from app.rag.embeddings import format_doc
def format_doc(doc):
    # NEW: handle metrics
    if doc.get("type") == "metric":
        return f"""
        Metric: {doc['name']}
        Definition: {doc['definition']}
        """

    # existing behavior (tables)
    return f"""
    Table: {doc['table']}
    Description: {doc['description']}
    Columns: {', '.join(doc['columns'])}
    """

metric_docs = []
for doc in raw_docs:
    if doc.get("type") == "metric":
        if doc["name"].lower() in question.lower():
            metric_docs.append(format_doc(doc))
        print('metric_docs:','\n',metric_docs)
        
# -------------------------
# 2. Vector search (tables)
# -------------------------
embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
).data[0].embedding

def generate_embeddings(docs):
    texts = [format_doc(doc) for doc in docs]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    embeddings = [e.embedding for e in response.data]

    return list(zip(texts, embeddings))

##collection = get_collection()
    # from app.rag/vector_store.py import get_chroma_client()
client = chromadb.PersistentClient(path="./chroma_db")
    # from app.rag/vector_store.py import get_collection()
collection = client.get_or_create_collection(name='schema_docs')

n_results = 2 #top_k 
# results = query_collection(collection, embedding, n_results=top_k)
    # from app.rag/vector_store.py import query_collection(collection, query_embedding, n_results=2)
"""
Query similar documents.

Why:
- This is the core of RAG retrieval
- Finds relevant tables based on meaning (not keywords)
"""
results = collection.query(query_embeddings=embedding,n_results=n_results)
docs = results.get("documents", [[]])[0]
table_docs = [d for d in docs if "Table:" in d]

# -------------------------
# 3. Remove weak matches
# -------------------------
filtered_tables = []
for t in table_docs:
    if any(word in t.lower() for word in question.lower().split()):
        filtered_tables.append(t)
if not filtered_tables:
    filtered_tables = table_docs
# -------------------------
# 4. Final ordering
# -------------------------
retrieve_relevant_docs = metric_docs + filtered_tables
retrieve_relevant_docs


output of load_schema_docs: 
 [{'table': 'users', 'description': 'Stores user account information', 'columns': ['user_id', 'signup_date']}, {'table': 'athletes', 'description': 'Stores athlete profiles', 'columns': ['athlete_id', 'name', 'sport', 'team']}, {'table': 'trades', 'description': 'Records all trades made by users for athletes', 'columns': ['trade_id', 'user_id', 'athlete_id', 'amount', 'created_at']}, {'type': 'metric', 'name': 'average trades per user', 'definition': 'total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades'}]
metric_docs: 
 ['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ']


['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ',
 '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ']

## Summary of vectore_store + embeddings --> retriever

In [31]:
from app.rag.embeddings import load_schema_docs, generate_embeddings
from app.rag.vector_store import get_collection, store_embeddings, query_collection, get_chroma_client
from openai import OpenAI

# --- store ---
docs = load_schema_docs()
embedded_docs = generate_embeddings(docs) 

collection = get_collection()
store_embeddings(collection, embedded_docs)

client = get_chroma_client()
# persist(client)

# --- query test ---
client_openai = OpenAI()

query = "average trades per user"

q_embed = client_openai.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

collection = get_collection()
results = query_collection(collection, q_embed)

results["documents"]

[['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ',
  '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ']]

In [32]:
results

{'ids': [['id_3', 'id_2']],
 'embeddings': None,
 'documents': [['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ',
   '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.6059892177581787, 0.9307858943939209]]}

In [ ]:
# Collection: 
collection.get()
# {'ids': ['id_0', 'id_1', 'id_2', 'id_3'],
#  'embeddings': None,
#  'documents': ['\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ',
#   '\n    Table: athletes\n    Description: Stores athlete profiles\n    Columns: athlete_id, name, sport, team\n    ',
#   '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
#   '\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        '],
#  'uris': None,
#  'included': ['metadatas', 'documents'],
#  'data': None,
#  'metadatas': [None, None, None, None]}

collection.get(include=["embeddings", "documents", "metadatas"])
# {'ids': ['id_0', 'id_1', 'id_2', 'id_3'],
#  'embeddings': array([[-0.00385475,  0.01408386,  0.00840759, ..., -0.00386238,
#           0.00679398, -0.01863098],
#         [ 0.0001303 ,  0.00159931,  0.00087214, ...,  0.01350403,
#           0.03649902,  0.00451279],
#         [-0.01150513,  0.00397491, -0.00366402, ...,  0.00390053,
#           0.00604248, -0.01422882],
#         [ 0.00627136,  0.00335312, -0.01785278, ..., -0.00287437,
#           0.01081085,  0.00393677]]),
#  'documents': ['\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ',
#   '\n    Table: athletes\n    Description: Stores athlete profiles\n    Columns: athlete_id, name, sport, team\n    ',
#   '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
#   '\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        '],
#  'uris': None,
#  'included': ['embeddings', 'documents', 'metadatas'],
#  'data': None,
#  'metadatas': [None, None, None, None]}

client = OpenAI()
docs = load_schema_docs()
embedded_docs = generate_embeddings(docs)

# This is how embedded docs look like.
import json
with open('embedded_docs_v1.json', 'w') as f:
    json.dump(embedded_docs, f, indent=4)

{'ids': ['id_0', 'id_1', 'id_2', 'id_3'],
 'embeddings': array([[-0.00387955,  0.01409149,  0.00841522, ..., -0.00386047,
          0.00680923, -0.01870728],
        [ 0.00012338,  0.00161648,  0.00090361, ...,  0.01348114,
          0.03649902,  0.00446701],
        [-0.01140594,  0.00441742, -0.00323296, ...,  0.00396347,
          0.00639343, -0.01371765],
        [ 0.00629807,  0.00337982, -0.0178833 , ..., -0.00285339,
          0.01080322,  0.00390625]]),
 'documents': ['\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ',
  '\n    Table: athletes\n    Description: Stores athlete profiles\n    Columns: athlete_id, name, sport, team\n    ',
  '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
  '\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; comp

## Context Builder

In [22]:
docs

[{'table': 'users',
  'description': 'Stores user account information',
  'columns': ['user_id', 'signup_date']},
 {'table': 'athletes',
  'description': 'Stores athlete profiles',
  'columns': ['athlete_id', 'name', 'sport', 'team']},
 {'table': 'trades',
  'description': 'Records all trades made by users for athletes',
  'columns': ['trade_id', 'user_id', 'athlete_id', 'amount', 'created_at']},
 {'type': 'metric',
  'name': 'average trades per user',
  'definition': 'total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades'}]

In [26]:

# app/rag/context_builder.py
# Build Clean Schema Context for the LLM
# OUTPUT: Formatted Schema Context

def build_context(docs: list) -> str:
    """
    Input: retrieved documents
    Output: formatted schema context for LLM

    Why:
    - LLM performs better with clean structured text
    - Avoid passing raw JSON or messy strings
    """

    context = "Relevant Database Schema:\n"

    for doc in docs:
        context += f"{doc.strip()}\n\n"

    return context.strip()
build_context(documents)

NameError: name 'documents' is not defined

In [50]:
docs

['\n        Metric: average trades per user\n        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades\n        ',
 '\n    Table: trades\n    Description: Records all trades made by users for athletes\n    Columns: trade_id, user_id, athlete_id, amount, created_at\n    ',
 '\n    Table: users\n    Description: Stores user account information\n    Columns: user_id, signup_date\n    ']

In [51]:
# docs = retrieve_relevant_docs(question=question,top_k=4)
context = "Relevant Database Schema:\n"
for doc in docs:
        # print(doc)
        # print(f"{doc.strip()}\n\n")
        context += f"{doc.strip()}\n\n"
print('Output of context builder:')
print(context)

Output of context builder:
Relevant Database Schema:
Metric: average trades per user
        Definition: total number of trades divided by number of unique users; computed as AVG(trade_count) where trade_count is COUNT(*) per user_id from trades

Table: trades
    Description: Records all trades made by users for athletes
    Columns: trade_id, user_id, athlete_id, amount, created_at

Table: users
    Description: Stores user account information
    Columns: user_id, signup_date




## End of Semanitc Layer 
##### The out put of all of this is relevant tables that will be used on in the next step to create SQL for the main query.

# SQL Generation & Execution

## SQL Geneation

In [ ]:
def generate_sql(user_query: str) -> str:
    # Step 1: Retrieve relevant schema docs
    docs = retrieve_relevant_docs(user_query)

    # Step 2: Build clean context
    context = build_context(docs)

    # Step 3: Construct prompt
    prompt = f"""
    {context}

    User Question:
    {user_query}
    """

    # Step 4: Call LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw_sql = response.choices[0].message.content
    cleaned_sql = clean_sql(raw_sql)

    print("\n--- RETRIEVED CONTEXT ---")
    print(context)
    print("------------------------\n")

    return cleaned_sql